### Notebook scope

### Purpose
Analyze final warming and pathway skill and generate final synthesis outputs.

### Scientific question
Does interactive O3 delay FWD and change O3/vortex pathway skill?

### Method
Use U60N50 FWD threshold definition and paired INT/NOCOUPL cases. Also assemble a final mechanism robustness matrix from available output tables.

### Expected output
FWD figures, skill figures, final robustness summary CSV/figure, and EXTENTION_ANALYSIS_FIGURE_GUIDE.md.

### Interpretation
Negative or positive skill deltas should be interpreted as changed pathway representation, not automatic improvement.

### Caveat
FWD may be NA when U60N50 never stays below threshold for 10 days within available lead time.


### Setup imports and shared paths

### Purpose
Load the shared Extention_analysis utility module and create output directories.

### Scientific question
Can every notebook start from a clean kernel and still find the same data roots and output roots?

### Method
Use DATA_ROOT, HINDCAST_ROOT, BWCN_ROOT, B2000_ROOT, WORK_ROOT, FIG_DIR, TAB_DIR, CACHE_DIR, and LOG_DIR from hindcast_extension_utils.py. No diagnostics are calculated in this cell.

### Expected output
Printed path check only. No figure is generated by this setup cell.

### Interpretation
Successful execution means the notebook can access the common utilities and write outputs in the agreed directory tree.

### Caveat
This setup does not prove that every downstream data product exists; missing products are logged later.


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

import hindcast_extension_utils as hx
from hindcast_extension_utils import *
_assign_member_short = hx._assign_member_short

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)

### Compute FWD distributions and skill metrics

### Purpose
Build INT/NOCOUPL FWD and skill comparison tables.

### Scientific question
Does INT delay FWD or improve/change O3 and vortex pathway skill relative to NOCOUPL?

### Method
FWD is first day U60N50 < 7 m/s for at least 10 days. Skill uses RMSE relative to BWCN same-year reference from init date to May30.

### Expected output
outputs/tables/08_FWD_INT_NOCOUPL_summary.csv, outputs/tables/08_INT_vs_NOCOUPL_skill_metrics.csv, and four figures.

### Interpretation
A positive INT-NOCOUPL FWD delta means INT delays final warming. Lower RMSE deltas mean smaller errors, but physical interpretation should consider persistence and feedback.

### Caveat
Pairs require same year/init small perturbation. Unpaired cases are not forced into this comparison.


In [ ]:
inv = discover_hindcast_cases(); pairs = paired_int_nocoupl_cases(inv)
fwd_rows = []; skill_rows = []
for _, pair in pairs.iterrows():
    for cfg, case in [("INT", pair["INT_case"]), ("NOCOUPL", pair["NOCOUPL_case"] )]:
        meta = parse_case_name(case)
        u50, _ = compute_u60(case, 50)
        fwd = compute_fwd_from_u60n50(u50)
        if not fwd.empty:
            fwd["case"] = case; fwd["config"] = cfg; fwd["year"] = meta["year"]; fwd["init_month"] = meta["init_month"]
            fwd_rows.append(fwd)
        o3, date = load_hindcast_o3(case); ref, ref_date = load_bwcn_reference_o3(meta["year"])
        rmse_o3 = compute_o3_rmse(o3, ref, *target_window_for_case(case))
        if u50 is not None:
            # Reference wind skill is intentionally left as NA until a same-year BWCN U60 reference is generated.
            pass
        if not rmse_o3.empty:
            ma = o3_ma_min(o3, date) if o3 is not None else pd.DataFrame()
            ref_ma = o3_ma_min(ref, ref_date) if ref is not None else pd.DataFrame()
            ref_min = float(ref_ma["O3_MA_min"].iloc[0]) if not ref_ma.empty else np.nan
            df = rmse_o3.merge(ma, on="member", how="left") if not ma.empty else rmse_o3
            df["O3_min_error"] = df.get("O3_MA_min", np.nan) - ref_min
            df["case"] = case; df["config"] = cfg; df["year"] = meta["year"]; df["init_month"] = meta["init_month"]
            skill_rows.append(df)
fwd_all = pd.concat(fwd_rows, ignore_index=True) if fwd_rows else pd.DataFrame()
skill = pd.concat(skill_rows, ignore_index=True) if skill_rows else pd.DataFrame()
fwd_csv = TAB_DIR / "08_FWD_INT_NOCOUPL_summary.csv"; skill_csv = TAB_DIR / "08_INT_vs_NOCOUPL_skill_metrics.csv"
fwd_all.to_csv(fwd_csv,index=False); skill.to_csv(skill_csv,index=False)
fig, axes = plt.subplots(1,2,figsize=(13,5),constrained_layout=True)
if fwd_all.empty:
    for ax in axes: ax.axis("off"); axes[0].text(0.5,0.5,"No FWD pair data",ha="center")
else:
    labels = [f"{y}-{m}" for y,m in fwd_all[["year","init_month"]].drop_duplicates().itertuples(index=False)]
    for i, ((year, init), sub) in enumerate(fwd_all.groupby(["year","init_month"])):
        for offset, cfg, color in [(-0.15,"INT","tab:red"),(0.15,"NOCOUPL","tab:blue")]:
            vals = sub.loc[sub["config"]==cfg,"FWD_DOY"].dropna()
            axes[0].scatter(np.full(len(vals), i+offset), vals, label=cfg if i==0 else None, alpha=0.75, color=color, edgecolor="black")
        piv = sub.pivot_table(index="member", columns="config", values="FWD_DOY")
        if {"INT","NOCOUPL"}.issubset(piv.columns):
            delta = piv["INT"] - piv["NOCOUPL"]
            axes[1].scatter(np.full(len(delta.dropna()), i), delta.dropna(), color="tab:purple", edgecolor="black")
    axes[0].set_xticks(range(len(labels)), labels, rotation=45, ha="right"); axes[0].set_ylabel("FWD DOY"); axes[0].legend(); axes[0].set_title("FWD distributions")
    axes[1].axhline(0,color="0.3",lw=0.8); axes[1].set_xticks(range(len(labels)), labels, rotation=45, ha="right"); axes[1].set_ylabel("INT - NOCOUPL FWD (days)"); axes[1].set_title("Paired FWD delay")
savefig(fig, "08_FWD_distributions_INT_vs_NOCOUPL", notebook="08_final_warming_and_skill.ipynb", scientific_question="Does INT delay final warming relative to NOCOUPL?", variables_windows="FWD from U60N50 < 7 m/s sustained 10 days", interpretation="Positive INT-NOCOUPL FWD deltas mean interactive O3 delays breakdown.", caveat="FWD is unavailable if the threshold is not crossed within the hindcast window.", csv_table=fwd_csv)
plt.show()
fig, axes = plt.subplots(1,2,figsize=(13,5),constrained_layout=True)
if skill.empty:
    for ax in axes: ax.axis("off"); axes[0].text(0.5,0.5,"No skill metrics",ha="center")
else:
    skill["pair"] = skill["year"] + "-" + skill["init_month"]
    for ax, col, title in [(axes[0], "O3_RMSE", "O3 RMSE"), (axes[1], "O3_min_error", "O3 MA minimum error")]:
        for i, pair_label in enumerate(sorted(skill["pair"].unique())):
            sub = skill[skill["pair"]==pair_label]
            for offset, cfg, color in [(-0.15,"INT","tab:red"),(0.15,"NOCOUPL","tab:blue")]:
                vals = sub.loc[sub["config"]==cfg, col].dropna()
                ax.scatter(np.full(len(vals), i+offset), vals, color=color, label=cfg if i==0 else None, alpha=0.75, edgecolor="black")
        ax.set_xticks(range(skill["pair"].nunique()), sorted(skill["pair"].unique()), rotation=45, ha="right"); ax.set_ylabel(col); ax.set_title(title)
    axes[0].legend()
savefig(fig, "08_INT_vs_NOCOUPL_skill_distributions", notebook="08_final_warming_and_skill.ipynb", scientific_question="Does INT improve or change O3 pathway skill?", variables_windows="O3 RMSE and March-April minimum error; init-May30", interpretation="Lower INT distributions indicate smaller O3 pathway error; sign should be interpreted with feedback persistence.", caveat="Skill is relative to BWCN same-year reference and does not imply Marina reproduction.", csv_table=skill_csv)
plt.show()

# Delta FWD versus reference ozone-depletion strength.
delta_rows = []
if not fwd_all.empty:
    sev_path = TAB_DIR / "01_reference_O3_severity_selected_years.csv"
    sev = pd.read_csv(sev_path) if sev_path.exists() else pd.DataFrame(columns=["year", "MA_min_O3_DU"])
    sev["year"] = sev["year"].astype(str).str.zfill(4) if not sev.empty else []
    for (year, init), sub in fwd_all.groupby(["year", "init_month"]):
        piv = sub.pivot_table(index="member", columns="config", values="FWD_DOY")
        if {"INT", "NOCOUPL"}.issubset(piv.columns):
            delta = piv["INT"] - piv["NOCOUPL"]
            sev_val = sev.loc[sev["year"] == str(year).zfill(4), "MA_min_O3_DU"]
            delta_rows.append({"year": str(year).zfill(4), "init_month": str(init).zfill(2), "delta_FWD_mean": float(delta.mean()), "delta_FWD_std": float(delta.std()), "reference_MA_min_O3_DU": float(sev_val.iloc[0]) if len(sev_val) else np.nan})
delta_fwd = pd.DataFrame(delta_rows)
delta_fwd_csv = TAB_DIR / "08_delta_FWD_vs_O3_depletion_strength.csv"
delta_fwd.to_csv(delta_fwd_csv, index=False)
fig, ax = plt.subplots(figsize=(6.8, 5.2), constrained_layout=True)
if delta_fwd.empty:
    ax.axis("off"); ax.text(0.5, 0.5, "No paired FWD deltas", ha="center")
else:
    ax.scatter(delta_fwd["reference_MA_min_O3_DU"], delta_fwd["delta_FWD_mean"], s=90, color="tab:purple", edgecolor="black")
    for _, row in delta_fwd.iterrows():
        ax.text(row["reference_MA_min_O3_DU"], row["delta_FWD_mean"], f"{row['year']}-{row['init_month']}", fontsize=8)
    ax.axhline(0, color="0.35", lw=0.8)
    ax.set_xlabel("Reference March-April minimum O3 (DU)")
    ax.set_ylabel("Mean INT - NOCOUPL FWD (days)")
    ax.set_title("Delta FWD vs O3 depletion strength")
savefig(fig, "08_delta_FWD_vs_O3_depletion_strength", notebook="08_final_warming_and_skill.ipynb", scientific_question="Is stronger reference ozone depletion associated with a larger INT FWD delay?", variables_windows="Reference MA minimum O3 and paired INT-NOCOUPL FWD delta", interpretation="Positive y values mean INT delays FWD; a trend with lower O3 indicates stronger feedback in deeper depletion events.", caveat="Only paired small-perturbation cases are included and sample size may be small.", csv_table=delta_fwd_csv)
plt.show()

# Skill difference heatmap.
skill_delta_rows = []
if not skill.empty:
    for (year, init), sub in skill.groupby(["year", "init_month"]):
        row = {"year": str(year).zfill(4), "init_month": str(init).zfill(2)}
        for col in ["O3_RMSE", "O3_min_error"]:
            piv = sub.pivot_table(index="member", columns="config", values=col)
            row[f"delta_{col}"] = float((piv["INT"] - piv["NOCOUPL"]).mean()) if {"INT", "NOCOUPL"}.issubset(piv.columns) else np.nan
        skill_delta_rows.append(row)
skill_delta = pd.DataFrame(skill_delta_rows)
skill_delta_csv = TAB_DIR / "08_INT_vs_NOCOUPL_skill_difference_heatmap.csv"
skill_delta.to_csv(skill_delta_csv, index=False)
fig, ax = plt.subplots(figsize=(8.5, max(3.8, 0.45 * len(skill_delta) + 2)), constrained_layout=True)
if skill_delta.empty:
    ax.axis("off"); ax.text(0.5, 0.5, "No paired skill deltas", ha="center")
else:
    mat = skill_delta.set_index(["year", "init_month"])[["delta_O3_RMSE", "delta_O3_min_error"]]
    im = ax.imshow(mat.values, cmap="RdBu_r", aspect="auto")
    ax.set_xticks(range(mat.shape[1]), mat.columns, rotation=30, ha="right")
    ax.set_yticks(range(mat.shape[0]), [f"{a}-{b}" for a, b in mat.index])
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat.iloc[i, j]
            ax.text(j, i, f"{val:.2g}" if np.isfinite(val) else "NA", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, label="INT - NOCOUPL skill metric")
    ax.set_title("INT vs NOCOUPL skill difference heatmap")
savefig(fig, "08_INT_vs_NOCOUPL_skill_difference_heatmap", notebook="08_final_warming_and_skill.ipynb", scientific_question="Does INT improve or change O3 pathway skill metrics relative to NOCOUPL?", variables_windows="Paired O3_RMSE and O3_min_error; init-May30", interpretation="Negative delta_O3_RMSE indicates smaller INT O3 RMSE; signs should be interpreted as pathway changes, not automatic improvement.", caveat="U60 and FWD skill deltas can be added after same-year BWCN wind references are generated.", csv_table=skill_delta_csv)
plt.show()


### Final robustness synthesis matrix

### Purpose
Generate the requested final mechanism robustness matrix and figure guide.

### Scientific question
Which parts of the 0008-01 mechanism are robust across cases, and which relate to INT feedback?

### Method
Read available tables from notebooks 01, 03, 04, 05, 07, and 08. Populate continuous values when available and NA when not applicable.

### Expected output
outputs/tables/final_mechanism_robustness_summary_matrix.csv, outputs/figures/final_mechanism_robustness_summary_matrix.png/pdf, and outputs/EXTENTION_ANALYSIS_FIGURE_GUIDE.md.

### Interpretation
Check marks indicate diagnostics supporting a link; question marks or gray NA show missing or not-applicable cases.

### Caveat
This synthesis is only as complete as the notebooks that have been run before it; rerun notebook 08 after running earlier notebooks for the fullest guide.


In [ ]:
inv = discover_hindcast_cases()
cases = inv["case"].tolist()
matrix = pd.DataFrame({"case": cases})
sev_path = TAB_DIR / "01_reference_O3_severity_selected_years.csv"
if sev_path.exists():
    sev = pd.read_csv(sev_path)[["year","MA_min_O3_DU"]]
    sev["year"] = sev["year"].astype(str).str.zfill(4)
    inv_year = inv[["case", "year"]].copy()
    inv_year["year"] = inv_year["year"].astype(str).str.zfill(4)
    matrix = matrix.merge(inv_year.merge(sev,on="year",how="left")[["case","MA_min_O3_DU"]], on="case", how="left")
else:
    matrix["MA_min_O3_DU"] = np.nan
for col in ["EP100_O3_link_R", "W1W2_dominance_R", "spread_order_score", "Z300_stationary_source_R", "Z300_wave2_source_R", "INT_delays_FWD_days", "INT_changes_O3_skill_delta"]:
    matrix[col] = np.nan
p03 = TAB_DIR / "03_wave_component_relevance_all_cases.csv"
if p03.exists():
    w = pd.read_csv(p03)
    for _, row in w.iterrows():
        if row["metric"] == "wave1_plus_wave2 vs O3_RMSE": matrix.loc[matrix["case"]==row["case"], "EP100_O3_link_R"] = row["R"]
        if row["metric"] == "all_waves vs W1+W2": matrix.loc[matrix["case"]==row["case"], "W1W2_dominance_R"] = row["R"]
p04 = TAB_DIR / "04_spread_onset_timing_summary_all_cases.csv"
if p04.exists():
    s = pd.read_csv(p04).pivot_table(index="case", columns="variable", values="onset_doy")
    for case in s.index:
        dyn = np.nanmin([s.loc[case].get("EP100_W1W2", np.nan), s.loc[case].get("U60N10", np.nan), s.loc[case].get("U60N50", np.nan)])
        o3 = s.loc[case].get("O3", np.nan)
        matrix.loc[matrix["case"]==case, "spread_order_score"] = o3 - dyn if np.isfinite(o3) and np.isfinite(dyn) else np.nan
p05 = TAB_DIR / "05_Z300_source_metric_correlations_all_cases.csv"
if p05.exists():
    z = pd.read_csv(p05)
    for _, row in z.iterrows():
        rel = row["relationship"]
        if "stationary_projection vs EP100_wave1_plus_wave2" in rel: matrix.loc[matrix["case"]==row["case"], "Z300_stationary_source_R"] = row["R"]
        if "Z300_wave2_amplitude vs EP100_wave2" in rel: matrix.loc[matrix["case"]==row["case"], "Z300_wave2_source_R"] = row["R"]
p08 = TAB_DIR / "08_FWD_INT_NOCOUPL_summary.csv"
if p08.exists():
    f = pd.read_csv(p08)
    if not f.empty:
        for (year, init), sub in f.groupby(["year","init_month"]):
            piv = sub.pivot_table(index="member", columns="config", values="FWD_DOY")
            if {"INT","NOCOUPL"}.issubset(piv.columns):
                delta = float((piv["INT"]-piv["NOCOUPL"]).mean())
                mask = (inv["year"]==str(year).zfill(4)) & (inv["init_month"]==str(init).zfill(2))
                matrix.loc[matrix["case"].isin(inv.loc[mask,"case"]), "INT_delays_FWD_days"] = delta
p08s = TAB_DIR / "08_INT_vs_NOCOUPL_skill_metrics.csv"
if p08s.exists():
    sk = pd.read_csv(p08s)
    if not sk.empty:
        for (year, init), sub in sk.groupby(["year","init_month"]):
            piv = sub.pivot_table(index="member", columns="config", values="O3_RMSE")
            if {"INT","NOCOUPL"}.issubset(piv.columns):
                delta = float((piv["INT"]-piv["NOCOUPL"]).mean())
                mask = (inv["year"]==str(year).zfill(4)) & (inv["init_month"]==str(init).zfill(2))
                matrix.loc[matrix["case"].isin(inv.loc[mask,"case"]), "INT_changes_O3_skill_delta"] = delta
out_csv = TAB_DIR / "final_mechanism_robustness_summary_matrix.csv"
matrix.to_csv(out_csv,index=False)
fig, ax = plt.subplots(figsize=(13, max(5, 0.32*len(matrix)+2)), constrained_layout=True)
plot_cols = [c for c in matrix.columns if c != "case"]
vals = matrix[plot_cols].astype(float).values
im = ax.imshow(vals, cmap="RdBu_r", aspect="auto")
ax.set_xticks(range(len(plot_cols)), plot_cols, rotation=45, ha="right"); ax.set_yticks(range(len(matrix)), matrix["case"])
for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        val = vals[i,j]
        if np.isfinite(val):
            txt = "✓" if ((plot_cols[j].endswith("_R") and abs(val)>=0.5) or (plot_cols[j]=="spread_order_score" and val>0) or (plot_cols[j]=="INT_delays_FWD_days" and val>0)) else f"{val:.2g}"
        else:
            txt = "?"
        ax.text(j,i,txt,ha="center",va="center",fontsize=8)
fig.colorbar(im, ax=ax, label="value")
ax.set_title("Final mechanism robustness summary matrix")
savefig(fig, "final_mechanism_robustness_summary_matrix", notebook="08_final_warming_and_skill.ipynb", scientific_question="Which parts of the 0008-01 mechanism are robust across hindcast cases?", variables_windows="Compiled diagnostics from reference severity, EP100, spread timing, Z300, INT/NOCOUPL, and FWD notebooks", interpretation="Check marks mark supportive diagnostics; question marks mark missing or not applicable evidence.", caveat="Run all notebooks before interpreting this synthesis as complete.", csv_table=out_csv)
plt.show()
guide = write_figure_guide()
with open(OUT_ROOT / "EXTENTION_ANALYSIS_SYNTHESIS.md", "w") as f:
    f.write("""# Extention Analysis Synthesis\n\n0008-01 provides the detailed source mechanism:\nZ300 stationary-wave geometry / wave-2 amplitude\n-> constructive or destructive interference\n-> EP100 W1+W2 anomaly\n-> U60N10 / U60N50 vortex pathway spread\n-> later O3 RMSE.\n\nThe remaining hindcasts test which parts of this chain are robust across event years and which parts are specifically associated with interactive-O3 feedback. Later-initialized cases should not be interpreted as long-lead precursor tests; they are better suited for diagnosing early post-initialization wave forcing, vortex persistence, final warming, and INT-vs-NOCOUPL feedback differences.\n""")
print("Wrote", guide)